In [20]:
import getpass
import json
import requests

BASE_URL = "https://bhoonidhi.nrsc.gov.in"
LOGIN_URL = f"{BASE_URL}/bhoonidhi/LoginServlet"


def compact_post(url: str, payload: dict, headers: dict) -> requests.Response:
    # WAF blocks JSON bodies containing whitespace — must serialize compact.
    body = json.dumps(payload, separators=(",", ":"))
    return requests.post(url, data=body, headers=headers, timeout=30)


def login(username: str, password: str) -> dict:
    headers = {
        "Host": "bhoonidhi.nrsc.gov.in",
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "Referer": f"{BASE_URL}/bhoonidhi/login.html",
        "Content-Type": "application/json",
        "X-Requested-With": "XMLHttpRequest",
        "Origin": BASE_URL,
    }
    payload = {
        "userId": username,
        "password": password,
        "action": "VALIDATE_LOGIN",
        "oldDB": "false",
    }
    resp = compact_post(LOGIN_URL, payload, headers)
    resp.raise_for_status()
    results = resp.json().get("Results") or []
    if not results or not results[0].get("JWT"):
        raise RuntimeError(f"Login failed: {results}")
    return results[0]

import os
from dotenv import load_dotenv

load_dotenv()

session = login(os.getenv("BHOONIDHI_USERNAME"), os.getenv("BHOONIDHI_PASSWORD"))

In [ ]:
from rich.console import Console
console = Console()
console.print(session)

{
    'MSG': 'ENABLED SERVICES:ONL_geovicco',
    'JWT': 
'eyJhbGciOiJIUzI1NiJ9.eyJzdWIiOiJBdXRoIiwiVXNlcklEIjoiT05MX2dlb3ZpY2NvIiwiSVBBZGRyZXNzIjoiMTEyLjIxMy4xNzMuMTUwIiwiU
2Vzc2lvbklEIjozOTgsImV4cGlyZXNBdFRpbWUiOiIyMDI2LTA4LTEwIDE2OjMyOjQ3In0.k7GcieIGiuE7pGsL4F8yJAaIwuDOe85gbD3VCYrHA1M'
,
    'USERCATNAME': 'PRIVATE',
    'USEREMAIL': 'geovicco@gmail.com',
    'TnC_ACC': 'Y',
    'USERID': 'ONL_geovicco',
    'GSTINFO': 'No',
    'USERNAME': 'ADITYA SHARMA',
    'NUser': 'N',
    'GE_NGE': 'NGE',
    'USERCAT': '04',
    'CARTCOUNT': '0'
}

In [3]:
import webbrowser

INDEX_URL = f"{BASE_URL}/bhoonidhi/index.html"


def open_portal(jwt: str | None = None) -> None:
    url = f"{INDEX_URL}?token={jwt}" if jwt else INDEX_URL
    webbrowser.open(url)

In [4]:
open_portal(session.get("JWT"))

### View Cart

Before searching lets see if any existing scenes are added to the cart

In [22]:
from zoneinfo import ZoneInfo
from datetime import datetime

IST = ZoneInfo("Asia/Kolkata")

CART_URL = f"{BASE_URL}/bhoonidhi/CartServlet"

def view_cart(jwt: str, user_id: str, cart_date: datetime | None = None) -> dict:
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json",
        "token": jwt,
    }
    cart_date = cart_date or datetime.now(IST)
    formatted_date = cart_date.strftime("%d %B %Y").replace(" ", "%20")
    payload = {"userId": user_id, "cartDate": formatted_date, "action": "VIEWCART"}
    resp = compact_post(CART_URL, payload, headers)
    resp.raise_for_status()
    return resp.json()


# if __name__ == "__main__":
print("cartDate (IST):", datetime.now(IST).strftime("%d %B %Y"))
cart = view_cart(session["JWT"], session["USERID"])
console.print(cart)

cartDate (IST): 10 August 2026


{'Results': []}

### Search for Scenes

In [23]:
from datetime import datetime, timedelta

SEARCH_URL = f"{BASE_URL}/bhoonidhi/ProductSearch"
ARCHIVE_URL = f"{BASE_URL}/bhoonidhi/SatSenServlet"


def get_archive_manifest() -> list[dict]:
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    payload = {"userId": "T", "action": "GETAVCONFIG", "userEmail": "abc@xyz.com"}
    resp = compact_post(ARCHIVE_URL, payload, headers)
    resp.raise_for_status()
    return resp.json()["Results"]


def search_scenes(
    sensor_disp_name: str,
    minx: float,
    miny: float,
    maxx: float,
    maxy: float,
    start_date: datetime,
    end_date: datetime,
) -> list[dict]:
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    sdate = start_date.strftime("%b%%2F%d%%2F%Y").upper()
    edate = end_date.strftime("%b%%2F%d%%2F%Y").upper()
    payload = {
        "userId": "T",
        "prod": "Standard",
        "selSats": sensor_disp_name,
        "offset": "0",
        "sdate": sdate,
        "edate": edate,
        "query": "area",
        "queryType": "polygon",
        "isMX": "No",
        "tllat": maxy,
        "tllon": minx,
        "brlat": miny,
        "brlon": maxx,
        "filters": "%7B%7D",
    }
    resp = compact_post(SEARCH_URL, payload, headers)
    resp.raise_for_status()
    return resp.json().get("Results", [])


# Search for Scenes
manifest = get_archive_manifest()
sat = next(m for m in manifest if m["satName"] == "ResourceSat-2A")
sensor = next(s for s in sat["sensors"] if "AWIF" in s["senName"])
disp_name = sensor.get("dispName") or f"{sat['satName']}_{sensor['senName']}"
scenes = search_scenes(
    sensor_disp_name=disp_name,
    minx=77.0,
    miny=12.0,
    maxx=78.5,
    maxy=13.5,
    start_date=datetime.now() - timedelta(days=30),
    end_date=datetime.now(),
)
print(f"Found {len(scenes)} scenes")

Found 9 scenes


In [24]:
# Only get scenes that are direct download
open_scenes = [sc for sc in scenes if sc.get("PRICED") == "OpenData_DirectDownload"]
console.print(f"Found {len(open_scenes)} open scenes")
console.print(open_scenes[0])

Found 9 open scenes

{
    'ID': 'RAW18JUL2026049872010400062PSANSTLCSRHTDC',
    'FILENAME': 'RAW18JUL2026049872010400062PSANSTLCSRHTDC',
    'DIRPATH': '/imgarchive/PRODUCTJPGS//R2A/AWIF/2026/JUL/18/',
    'IMAGING_ORBIT_NO': '049872',
    'GROUND_ORBIT_NO': '049872',
    'SESSION_NO': '1',
    'ROLL': '0.000000',
    'PITCH': '-999',
    'YAW': '-999',
    'PATHNO': '104',
    'SCENE_NO': '62',
    'COVERAGE': 'NA',
    'CURR_SCENE_NO': 'Y',
    'ImgCrnNWLat': '16.298721',
    'ImgCrnNWLon': '78.2683',
    'ImgCrnNELat': '16.300445',
    'ImgCrnNELon': '82.7997',
    'ImgCrnSELat': '12.211756',
    'ImgCrnSELon': '82.7609',
    'ImgCrnSWLat': '12.210062',
    'ImgCrnSWLon': '78.3104',
    'IMAGE_CHAIN': 'NA',
    'AGENCY': 'NA',
    'SCENE_SEQ': '0',
    'OverLapPercent': '1.37855',
    'CrnNWLat': '16.298721',
    'CrnNWLon': '78.2683',
    'CrnNELat': '16.300445',
    'CrnNELon': '82.7997',
    'CrnSELat': '12.211756',
    'CrnSELon': '82.7609',
    'CrnSWLat': '12.210062',
    'CrnSWLon': '78.3104',
    'SCENE_CENTER_LAT': '14.265989',
    'SCENE_CENTER_LONG': '80.5348',
    'OBSID': '0',
    'ACQUISITION_MODE': '0',
    'IMAGING_MODE': '',
    'SATELLITE': 'R2A',
    'SENSOR': 'AWIF',
    'PRICED': 'OpenData_DirectDownload',
    'TABLETYPE': 'PMETA',
    'O2_MODE': '-',
    'DOP': '18-Jul-2026',
    'QUALITY_SCORE': 'Q',
    'PRODCODE': 'STLCSRHTD',
    'PRODTYPE': 'BOA-Archives',
    'BINPERIOD': '',
    'BINRESOLUTION': '',
    'PASS_TYPE': 'PLD',
    'QAZIP': 'NA',
    'QAPDF': 'NA',
    'srt': '20260810_FHZ012041',
    'SELECTION': 'ResourceSat-2A_AWIFS_BOA-Archives'
}

In [ ]:
def open_search_results(jwt: str | None = None) -> None:
    # index.html hosts Search-Criteria / Search-Results / Cart as tabs —
    # there's no separate results URL, just the same page + token.
    url = f"{INDEX_URL}?token={jwt}" if jwt else INDEX_URL
    webbrowser.open(url)

open_search_results(session["JWT"])

### Add Scenes to Cart

In [25]:
import urllib.parse

OPEN_ORDER_CART_URL = f"{BASE_URL}/bhoonidhi/OpenOrderCart"
PI_CART_URL = f"{BASE_URL}/bhoonidhi/PICartServlet"


def add_to_order_cart(scene: dict, jwt: str, user_id: str) -> dict:
    """For OpenData_OnOrder scenes (queryType=SMETA)."""
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json",
        "token": jwt,
    }
    payload = {
        "sceneID": scene["ID"],
        "srt": scene["srt"],
        "queryType": "SMETA",
        "action": "ADDTOORDERCART",
        "userId": user_id,
        "selProds": urllib.parse.quote(json.dumps(scene, separators=(",", ":"))),
        "selOtherProds": "NA",
        "selSats": scene["SELECTION"],
        "prod": "Standard",
    }
    resp = compact_post(OPEN_ORDER_CART_URL, payload, headers)
    resp.raise_for_status()
    return resp.json()


def add_to_pi_cart(scene: dict, jwt: str, user_id: str) -> dict:
    """For Priced scenes (queryType=TMETA)."""
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json",
        "token": jwt,
    }
    payload = {
        "sceneID": scene["ID"],
        "srt": scene["srt"],
        "queryType": "TMETA",
        "action": "ADDTOPICART",
        "userId": user_id,
        "selProds": urllib.parse.quote(json.dumps(scene, separators=(",", ":"))),
        "selOtherProds": "NA",
        "selSats": scene["SELECTION"],
        "prod": "Standard",
    }
    resp = compact_post(PI_CART_URL, payload, headers)
    resp.raise_for_status()
    return resp.json()

def add_to_cart(scene: dict, jwt: str, user_id: str) -> dict:
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json",
        "token": jwt,
    }
    payload = {
        "dop": scene["DOP"],
        "PROD_ID": scene["ID"],
        "PROD_AV": "Y",
        "srt": scene["srt"],
        "selProds": urllib.parse.quote(json.dumps(scene, separators=(",", ":"))),
        "action": "ADDTOCART",
        "userId": user_id,
    }
    resp = compact_post(CART_URL, payload, headers)
    resp.raise_for_status()
    return resp.json()

In [30]:
CART_URL = f"{BASE_URL}/bhoonidhi/CartServlet"

for scene in open_scenes[1:2]:  # one at a time
    result = add_to_cart(scene, session["JWT"], session["USERID"])
    # result = add_to_order_cart(scene, session["JWT"], session["USERID"])
    print(scene["ID"], "->", result)

RAW17JUL2026049858009900062PSANSTLCSRHTDC -> {'Results': [{'MSG': 'SUCCESS', 'cartCount': '2'}]}


### View Cart

In [31]:
print("cartDate (IST):", datetime.now(IST).strftime("%d %B %Y"))
cart = view_cart(session["JWT"], session["USERID"])
console.print(cart)

cartDate (IST): 10 August 2026


{
    'Results': [
        {
            'IMAGE_CHAIN': 'NA',
            'CURR_SCENE_NO': 'Y',
            'OBSID': '0',
            'SRT_ID': '20260810_FHZ012041',
            'CrnSWLon': '73.045',
            'PRODCODE': 'STLCSRHTD',
            'PASS_TYPE': 'PLD',
            'STATUS': 'ADDED',
            'PRICED': 'OpenData_DirectDownload',
            'PATHNO': '99',
            'SCENE_NO': '62',
            'ID': 'RAW17JUL2026049858009900062PSANSTLCSRHTDC',
            'CrnNELat': '16.282889',
            'QUALITY_SCORE': 'Q',
            'BINPERIOD': '',
            'CrnNELon': '77.5287',
            'SCENE_CENTER_LAT': '14.28068',
            'O2_MODE': '-',
            'SCENE_CENTER_LONG': '75.267',
            'ImgCrnNWLon': '73.0034',
            'DOP': '17-Jul-2026',
            'ACQUISITION_MODE': '0',
            'PITCH': '-999',
            'CrnNWLat': '16.280924',
            'SENSOR': 'AWIF',
            'ImgCrnNELat': '16.282889',
            'ImgCrnSWLat': '12.257044',
            'ImgCrnSELon': '77.4907',
            'SELECTION': 'ResourceSat-2A_AWIFS_BOA-Archives',
            'IMAGING_MODE': '',
            'PRODTYPE': 'BOA-Archives',
            'ImgCrnNELon': '77.5287',
            'GROUND_ORBIT_NO': '049858',
            'PROD_AV': 'Y',
            'COVERAGE': 'NA',
            'TABLETYPE': 'PMETA',
            'PRODUCTID': 'RAW17JUL2026049858009900062PSANSTLCSRHTDC',
            'YAW': '-999',
            'ImgCrnSELat': '12.258975',
            'SESSION_NO': '1',
            'CrnSELat': '12.258975',
            'SCENE_SEQ': '0',
            'QAPDF': 'NA',
            'CrnSWLat': '12.257044',
            'ROLL': '0.000000',
            'FILENAME': 'RAW17JUL2026049858009900062PSANSTLCSRHTDC',
            'ImgCrnSWLon': '73.045',
            'SATELLITE': 'R2A',
            'QAZIP': 'NA',
            'CrnSELon': '77.4907',
            'DIRPATH': '/imgarchive/PRODUCTJPGS//R2A/AWIF/2026/JUL/17/',
            'AGENCY': 'NA',
            'srt': '20260810_FHZ012041',
            'DCOUNT': '0',
            'ImgCrnNWLat': '16.280924',
            'OverLapPercent': '3.41453',
            'BINRESOLUTION': '',
            'IMAGING_ORBIT_NO': '049858',
            'CrnNWLon': '73.0034'
        },
        {
            'IMAGE_CHAIN': 'NA',
            'CURR_SCENE_NO': 'Y',
            'SCENE_SPEC': '049872_104_62_C',
            'OBSID': '0',
            'SRT_ID': '20260810_O0P011242',
            'CrnSWLon': '78.3104',
            'SAT_SPEC_SCHEME': 'Satellite_Sensor_ImagingMode_Subscene_Product',
            'PRODCODE': 'STLCSRHTD',
            'PASS_TYPE': 'PLD',
            'STATUS': 'ADDED',
            'PRICED': 'OpenData_DirectDownload',
            'PATHNO': '104',
            'SCENE_NO': '62',
            'ID': 'RAW18JUL2026049872010400062PSANSTLCSRHTDC',
            'CrnNELat': '16.300445',
            'QUALITY_SCORE': 'Q',
            'BINPERIOD': '',
            'CrnNELon': '82.7997',
            'SCENE_CENTER_LAT': '14.265989',
            'O2_MODE': '-',
            'SCENE_CENTER_LONG': '80.5348',
            'SCENE_ID': 'RAW18JUL2026049872010400062PSANSTLCSRHTDC',
            'ImgCrnNWLon': '78.2683',
            'DOP': '18-Jul-2026',
            'ACQUISITION_MODE': '0',
            'PITCH': '-999',
            'CrnNWLat': '16.298721',
            'SENSOR': 'AWIF',
            'cartDate': '2026-08-10',
            'ImgCrnNELat': '16.300445',
            'ImgCrnSWLat': '12.210062',
            'SAT_SPEC': 'R2A_AWIF_-_S_BOA-Archives',
            'ImgCrnSELon': '82.7609',
            'SELECTION': 'ResourceSat-2A_AWIFS_BOA-Archives',
            'IMG_PATH': 
'/imgarchive/PRODUCTJPGS//R2A/AWIF/2026/JUL/18//RAW18JUL2026049872010400062PSANSTLCSRHTDC.jpg',
            'IMAGING_MODE': '-',
            'PRODTYPE': 'BOA-Archives',
            'ImgCrnNELon': '82.7997',
            'GROUND_ORBIT_NO': '049872',
            'PROD_AV': 'Y',
            'SCENE_SPEC_SCHEME': 'GroundOrbit_Path_Row_Subscene',
         

In [ ]:
# DELTE
# CONFIRM